<a href="https://colab.research.google.com/github/rishita-svg/google-colab/blob/main/day1_manuscriptsextract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch torchvision Pillow

In [4]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

# 1. PUT YOUR EXACT FILE NAME HERE:
image_name = "road of engineers_2.jpg"

print("Loading image...")
image = Image.open(image_name).convert("RGB")

print("Downloading the AI model (this takes a minute)...")
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

print("Extracting text...")
pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
extracted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*30)
print("--- EXTRACTED TEXT ---")
print("="*30)
print(extracted_text)

Loading image...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

...

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extracting text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1616: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(



--- EXTRACTED TEXT ---
0 0


In [5]:
!pip install -q accelerate torchvision
!pip install -q git+https://github.com/huggingface/transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image

# 1. Ensure your correct file name is here:
image_name = "road of engineers_2.jpg"

print("Loading image...")
image = Image.open(image_name).convert("RGB")

print("Loading Qwen2-VL-2B Model...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

# MEMORY OPTIMIZATION: Set explicit pixel limits to avoid CUDA Out-Of-Memory
# This prevents the image from generating too many visual tokens.
processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    min_pixels=256*256,
    max_pixels=512*512
)

# Formatting the visual chat template
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Please transcribe all the readable handwritten or printed text found in this historical document image accurately. Maintain the layout lines if possible."}
        ]
    }
]

print("Processing image and text prompt with pixel limits...")
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = processor(text=[text], images=image, padding=True, return_tensors="pt")
inputs = inputs.to("cuda")

print("The VLM is reading your manuscript within safe memory boundaries...")
# Added pad_token_id to keep the generation clean
generated_ids = model.generate(**inputs, max_new_tokens=512, pad_token_id=processor.tokenizer.pad_token_id)
generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

print("\n" + "="*30)
print("--- VLM EXTRACTED TEXT ---")
print("="*30)
print(output_text)

Loading image...
Loading Qwen2-VL-2B Model...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Processing image and text prompt with pixel limits...
The VLM is reading your manuscript within safe memory boundaries...

--- VLM EXTRACTED TEXT ---
(10,10),(988,988)


In [9]:
!pip install -q timm einops

In [2]:
import torch
import os
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. THE PATCH: Trick the library scanner into ignoring flash_attn
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

# 2. Ensure your correct file name is here:
image_name = "road of engineers_2.jpg"

print("Loading high-resolution image...")
image = Image.open(image_name).convert("RGB")

print("Loading Microsoft Florence-2-large (Bypassing flash_attn requirement)...")

# 3. Apply the patch silently while the model loads
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")

processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-large",
    trust_remote_code=True
)

prompt = "<OCR>"

print("Processing prompt and image...")
inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda")
inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

print("The VLM is transcribing the high-res document text...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3
    )

output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*30)
print("--- FLORENCE-2 EXTRACTED TEXT ---")
print("="*30)
print(output_text)

Loading high-resolution image...
Loading Microsoft Florence-2-large (Bypassing flash_attn requirement)...


model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Processing prompt and image...
The VLM is transcribing the high-res document text...

--- FLORENCE-2 EXTRACTED TEXT ---
-
-
O PREMIUM
LEAPER & SINGLE
LAPPER & SCALLE
CALIFORNIA
LAMPER & SCALE
LAMPER & SINCALES
CINN CO.
CAMPER SINGLES
CAMPER SINGLIES
PARKER
CIMA CO. CAMPER CO. LAMPS
LARGE
LADIES
RADIO
RODDING
RIDGESTONE
RACING
PRA BRIED
PALM GATEES
PARA BRIER
PIAA BRIES
SCALE/200/FITTO AN INCH.



In [11]:
!pip install -q transformers==4.41.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.4 MB/s eta 0:00:00


In [3]:
import torch
import os
import glob
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. Flash-Attention patch to ensure smooth loading
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

print("Initializing Florence-2-large...")
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")

processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Find all images in your current directory (.jpg, .jpeg, .png)
# Change path if your files are inside a specific folder (e.g., "/content/my_folder/*.jpg")
image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f"/content/{ext}"))

image_files = sorted(list(set(image_files)))
print(f"Found {len(image_files)} images to process.")

# 3. Open a text file to write the final results
output_report_path = "/content/transcription_results.txt"

with open(output_report_path, "w") as f:
    f.write("==================================================\n")
    f.write("         BATCH EXTRACTION REPORT                  \n")
    f.write("==================================================\n\n")

    for idx, img_path in enumerate(image_files, 1):
        filename = os.path.basename(img_path)
        print(f"[{idx}/{len(image_files)}] Processing {filename}...")

        try:
            # Load and process image
            image = Image.open(img_path).convert("RGB")
            inputs = processor(text="<OCR>", images=image, return_tensors="pt").to("cuda")
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    max_new_tokens=1024,
                    do_sample=False,
                    num_beams=3
                )

            output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

            # Save result to file
            f.write(f"--- FILE: {filename} ---\n")
            f.write(output_text + "\n")
            f.write("-" * 50 + "\n\n")

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            f.write(f"--- FILE: {filename} ---\n")
            f.write(f"ERROR DURING PROCESSING: {str(e)}\n")
            f.write("-" * 50 + "\n\n")

print(f"\nBatch processing complete! Your results are saved in: {output_report_path}")

Initializing Florence-2-large...
Found 1 images to process.
[1/1] Processing road of engineers_2.jpg...

Batch processing complete! Your results are saved in: /content/transcription_results.txt


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!unzip -q /content/manuscripts.zip -d /content/manuscripts_folder

unzip:  cannot find or open /content/manuscripts.zip, /content/manuscripts.zip.zip or /content/manuscripts.zip.ZIP.


In [4]:
# 1. Update the server's package list so it finds the newest version
!apt-get update

# 2. Install the PDF conversion tools
!apt-get install -y poppler-utils
!pip install -q pdf2image

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,046 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jamm

In [5]:
import torch
import os
import glob
from PIL import Image
from pdf2image import convert_from_path
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. Flash-Attention patch for Florence-2 environment handling
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

print("Initializing Florence-2-large pipeline...")
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Set your input directory path here!
# If you used Method 1 (Google Drive), use: "/content/drive/MyDrive/manuscripts"
# If you used Method 2 (Zip file), use: "/content/manuscripts_folder/manuscripts"
SOURCE_DIR = "/content/drive/MyDrive/manuscripts"
OUTPUT_REPORT = "/content/final_transcription_report.txt"

# Gather all files from the directory
all_files = glob.glob(os.path.join(SOURCE_DIR, "*"))
print(f"Total files found in directory: {len(all_files)}")

with open(OUTPUT_REPORT, "w") as report:
    report.write("==================================================\n")
    report.write("       MASTER MANUSCRIPT EXTRACTION REPORT        \n")
    report.write("==================================================\n\n")

    for idx, file_path in enumerate(all_files, 1):
        filename = os.path.basename(file_path)
        ext = filename.split('.')[-1].lower()

        # Skip hidden files or system files
        if filename.startswith('.'): continue

        print(f"[{idx}/{len(all_files)}] Processing: {filename}")
        report.write(f"=== START OF FILE: {filename} ===\n")

        try:
            # FRAMEWORK BRANCH A: If the file is a PDF
            if ext == 'pdf':
                print(f"-> Converting PDF pages to high-res images...")
                pages = convert_from_path(file_path, dpi=300)
                for page_num, page_img in enumerate(pages, 1):
                    inputs = processor(text="<OCR>", images=page_img, return_tensors="pt").to("cuda")
                    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                    with torch.no_grad():
                        generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                    report.write(f"[Page {page_num}]\n{text}\n")

            # FRAMEWORK BRANCH B: If the file is a standard Image (PNG, JPG, JPEG)
            elif ext in ['jpg', 'jpeg', 'png', 'tiff', 'bmp']:
                image = Image.open(file_path).convert("RGB")
                inputs = processor(text="<OCR>", images=image, return_tensors="pt").to("cuda")
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                with torch.no_grad():
                    generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                report.write(f"{text}\n")

            else:
                report.write(f"Skipped: Unsupported file extension .{ext}\n")

        except Exception as e:
            print(f"-> Error processing {filename}: {str(e)}")
            report.write(f"ERROR ENCOUNTERED: {str(e)}\n")

        report.write("-" * 60 + "\n\n")

print(f"\nProcessing complete! All transcriptions compiled into: {OUTPUT_REPORT}")

Initializing Florence-2-large pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total files found in directory: 11
[1/11] Processing: unknown map.jpg
[2/11] Processing: unknown road of engineer.jpg
[3/11] Processing: lighthouse description of alexander.pdf
-> Converting PDF pages to high-res images...
[4/11] Processing: bradford manuscript.pdf
-> Converting PDF pages to high-res images...
[5/11] Processing: wercefter june 13th 1774.pdf
-> Converting PDF pages to high-res images...
[6/11] Processing: Affidavits, Statements 1864_1.pdf
-> Converting PDF pages to high-res images...
[7/11] Processing: Affidavits, Statements 1864_2.pdf
-> Converting PDF pages to high-res images...
[8/11] Processing: 1897 Senate Bill 0226. Resolve Relative To The Publication Of Bradford's Manuscript History Of The Plymouth Plantation.pdf
-> Converting PDF pages to high-res images...
[9/11] Processing: road of engineers_2.jpg
[10/11] Processing: unknown map_2.jpg
[11/11] Processing: massachussets_1.pdf
-> Converting PDF pages to high-res images...

Processing complete! All transcriptions 

In [6]:
import torch
import os
import glob
from PIL import Image
from pdf2image import convert_from_path
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports
from unittest.mock import patch

# 1. Flash-Attention patch for Florence-2 environment handling
def fixed_get_imports(filename: str | os.PathLike) -> list[str]:
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

print("Initializing Florence-2-large pipeline...")
with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Florence-2-large",
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to("cuda")
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Set your input and output directories
# (Make sure SOURCE_DIR points to wherever your files are currently located!)
SOURCE_DIR = "/content/drive/MyDrive/manuscripts"
OUTPUT_DIR = "/content/extracted_texts"

# Create the output directory if it doesn't exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Gather all files from the directory
all_files = glob.glob(os.path.join(SOURCE_DIR, "*"))
print(f"Total files found in directory: {len(all_files)}\n")

for idx, file_path in enumerate(all_files, 1):
    filename = os.path.basename(file_path)
    ext = filename.split('.')[-1].lower()

    # Skip hidden files or folders
    if filename.startswith('.') or os.path.isdir(file_path):
        continue

    print(f"[{idx}/{len(all_files)}] Processing: {filename}")

    # 3. Create a matching text file name (e.g., "map.jpg" -> "map.txt")
    base_name = os.path.splitext(filename)[0]
    output_file_path = os.path.join(OUTPUT_DIR, f"{base_name}.txt")

    try:
        # Open the new specific text file to write into
        with open(output_file_path, "w", encoding="utf-8") as out_file:

            # FRAMEWORK BRANCH A: If the file is a PDF
            if ext == 'pdf':
                print(f"  -> Converting PDF pages...")
                pages = convert_from_path(file_path, dpi=300)
                for page_num, page_img in enumerate(pages, 1):
                    inputs = processor(text="<OCR>", images=page_img, return_tensors="pt").to("cuda")
                    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                    with torch.no_grad():
                        generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                    out_file.write(f"[Page {page_num}]\n{text}\n\n")

            # FRAMEWORK BRANCH B: If the file is a standard Image
            elif ext in ['jpg', 'jpeg', 'png', 'tiff', 'bmp']:
                image = Image.open(file_path).convert("RGB")
                inputs = processor(text="<OCR>", images=image, return_tensors="pt").to("cuda")
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

                with torch.no_grad():
                    generated_ids = model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=1024, num_beams=3)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                out_file.write(f"{text}\n")

            else:
                print(f"  -> Skipped: Unsupported file extension .{ext}")

    except Exception as e:
        print(f"  -> Error processing {filename}: {str(e)}")
        # If it crashes on a specific file, write the error into that text file
        with open(output_file_path, "w", encoding="utf-8") as err_file:
            err_file.write(f"ERROR ENCOUNTERED DURING EXTRACTION: {str(e)}\n")

print(f"\nProcessing complete! All 11 text files are saved in: {OUTPUT_DIR}")

Initializing Florence-2-large pipeline...
Total files found in directory: 11

[1/11] Processing: unknown map.jpg
[2/11] Processing: unknown road of engineer.jpg
[3/11] Processing: lighthouse description of alexander.pdf
  -> Converting PDF pages...
[4/11] Processing: bradford manuscript.pdf
  -> Converting PDF pages...
[5/11] Processing: wercefter june 13th 1774.pdf
  -> Converting PDF pages...
[6/11] Processing: Affidavits, Statements 1864_1.pdf
  -> Converting PDF pages...
[7/11] Processing: Affidavits, Statements 1864_2.pdf
  -> Converting PDF pages...
[8/11] Processing: 1897 Senate Bill 0226. Resolve Relative To The Publication Of Bradford's Manuscript History Of The Plymouth Plantation.pdf
  -> Converting PDF pages...
[9/11] Processing: road of engineers_2.jpg
[10/11] Processing: unknown map_2.jpg
[11/11] Processing: massachussets_1.pdf
  -> Converting PDF pages...

Processing complete! All 11 text files are saved in: /content/extracted_texts


In [7]:
!zip -r /content/all_transcriptions.zip /content/extracted_texts

  adding: content/extracted_texts/ (stored 0%)
  adding: content/extracted_texts/unknown map.txt (deflated 35%)
  adding: content/extracted_texts/lighthouse description of alexander.txt (deflated 55%)
  adding: content/extracted_texts/bradford manuscript.txt (deflated 56%)
  adding: content/extracted_texts/Affidavits, Statements 1864_1.txt (deflated 31%)
  adding: content/extracted_texts/wercefter june 13th 1774.txt (deflated 44%)
  adding: content/extracted_texts/massachussets_1.txt (deflated 52%)
  adding: content/extracted_texts/Affidavits, Statements 1864_2.txt (deflated 41%)
  adding: content/extracted_texts/1897 Senate Bill 0226. Resolve Relative To The Publication Of Bradford's Manuscript History Of The Plymouth Plantation.txt (deflated 58%)
  adding: content/extracted_texts/unknown road of engineer.txt (deflated 39%)
  adding: content/extracted_texts/road of engineers_2.txt (deflated 44%)
  adding: content/extracted_texts/unknown map_2.txt (deflated 29%)
